# CRIVO — Transformer próprio de 15.855.360 parâmetros
Pesos aleatórios próprios, vocabulário próprio; nenhum modelo pré-treinado é baixado.
Use **Ambiente de execução → Alterar tipo → GPU**. Execute as células na ordem.
O pré-treino longo usa textos públicos em português. Depois, o SFT usa diálogos humanos e demonstrações autorais identificadas. Não há garantia de conversa inteligente: comparar respostas reservadas é obrigatório.

Os pesos, Adam, RNG e relatórios são gravados diretamente no Drive. Se a sessão cair, reexecute as células: a revisão do código e a pasta anterior são recuperadas. Não apague `checkpoint.pt` ou `melhor/`.

In [ ]:
from google.colab import drive
from pathlib import Path
import datetime, json, subprocess, sys

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/CRIVO/transformer-16m')
BASE.mkdir(parents=True, exist_ok=True)
PONTEIRO = BASE / 'ultimo_experimento.json'
# False retoma o último experimento; True cria um novo, sem apagar o anterior.
NOVO_EXPERIMENTO = False
ROOT = Path('/content/CRIVO-transformer-16m')
def preparar_codigo(raiz, revisao, repositorio='https://github.com/HROSONE/CRIVO.git'):
    raiz = Path(raiz)
    if not (raiz / '.git').exists():
        subprocess.run(['git', 'clone', '--no-checkout', repositorio, str(raiz)], check=True)
    # Um clone --no-checkout ainda não tem index nem arquivos de trabalho.
    # Seu status mostra D para todos os arquivos: não são alterações do usuário.
    vazio = not any(p.name != '.git' for p in raiz.iterdir())
    index_vazio = not subprocess.run(['git', 'ls-files', '-z'], cwd=raiz,
                                    capture_output=True, check=True).stdout
    alteracoes = subprocess.run(['git', 'status', '--porcelain'], cwd=raiz,
                               capture_output=True, text=True, check=True).stdout
    if alteracoes.strip() and not (vazio and index_vazio):
        raise RuntimeError('O checkout contém alterações reais. Preserve-as antes de continuar.\n' + alteracoes)
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', revisao], cwd=raiz, check=True)
    subprocess.run(['git', 'checkout', '--no-overwrite-ignore', '--detach', 'FETCH_HEAD'], cwd=raiz, check=True)
    return subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=raiz,
                          capture_output=True, text=True, check=True).stdout.strip()

if PONTEIRO.exists() and not NOVO_EXPERIMENTO:
    registro = json.loads(PONTEIRO.read_text())
    REVISAO = registro['revisao']
    SAIDA = BASE / registro['experimento']
else:
    REVISAO = 'codex/transformer-16m'
    SAIDA = BASE / datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
REVISAO = preparar_codigo(ROOT, REVISAO)
SAIDA.mkdir(parents=True, exist_ok=True)
registro = dict(revisao=REVISAO, experimento=SAIDA.name)
PONTEIRO.write_text(json.dumps(registro, indent=2) + '\n')
(SAIDA / 'execucao_colab.json').write_text(json.dumps(registro, indent=2) + '\n')
print('Código:', REVISAO, '\nResultados:', SAIDA)

In [ ]:
# Preservar o PyTorch com CUDA já instalado pelo Colab.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers>=0.20,<1', 'numpy>=1.24,<3', 'pyarrow>=15,<26'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Este currículo longo precisa de GPU. Ative GPU e reexecute; o piloto CPU está no GitHub.')
print(torch.__version__, torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'scripts/experimento_transformer_16m.py', '--etapa', 'config'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'testes_transformer_16m', 'testes_linguagem_profunda', 'testes_conversa_gerativa'], cwd=ROOT, check=True)

In [ ]:
# Somente dados públicos em revisões fixas, conferidos por SHA-256.
FONTES = Path('/content/crivo-fontes-16m')
CORPUS = Path('/content/crivo-corpus-16m')
FONTES.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-c', "from pathlib import Path; from scripts.baixar_fontes_linguagem import baixar; p=Path('/content/crivo-fontes-16m'); baixar('oasst2',p); baixar('wikipedia',p)"], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/preparar_conversa_gerativa.py', '--oasst2', str(FONTES/'oasst2.messages.jsonl.gz'), '--wikipedia', str(FONTES/'wikipedia.pt.parquet'), '--saida', str(CORPUS)], cwd=ROOT, check=True)
import shutil
shutil.copyfile(CORPUS/'manifesto.json', SAIDA/'corpus_manifesto.json')

## Pré-treino do zero: execute e retome esta célula até completar
O horizonte é 60.000 passos de lote 12 (até 184,32 milhões de tokens-alvo). A célula pausa após duas horas, preservando os estados. A duração real depende da GPU. A pausa por tempo não significa que o currículo terminou.

Se o Colab interromper a sessão, reabra este notebook e execute novamente desde o início. O ponteiro no Drive recupera a revisão e o experimento. Para parar cedo com segurança, use a interrupção depois de um checkpoint salvo; ele é gravado a cada 100 passos.

In [ ]:
subprocess.run([sys.executable, '-u', 'scripts/experimento_transformer_16m.py', '--etapa', 'pretreino', '--corpus', str(CORPUS), '--saida', str(SAIDA), '--dispositivo', 'cuda', '--max-segundos', '7200'], cwd=ROOT, check=True)
r = json.loads((SAIDA/'pretreino/relatorio.json').read_text())
print('Passos:', r['passo'], '/', r['horizonte'], '| Tokens aprendidos:', r['tokens_alvo'])
if r['passo'] < r['horizonte']:
    print('Pré-treino pausado. Execute esta célula novamente para retomar antes do SFT.')

## Ajuste de conversa
Esta etapa utiliza o melhor checkpoint do pré-treino próprio 16M. Metade dos exemplos de cada lote é humana; os demais são exercícios autorais. O melhor checkpoint é escolhido somente pela perda dos diálogos humanos de **validação**, sem usar o teste final. A perda mede previsão de tokens e não comprova compreensão.

In [ ]:
r = json.loads((SAIDA/'pretreino/relatorio.json').read_text())
if r['passo'] < r['horizonte']:
    raise RuntimeError('O pré-treino longo ainda está pausado. Retome a célula anterior até completar os 60.000 passos.')
subprocess.run([sys.executable, '-u', 'scripts/experimento_transformer_16m.py', '--etapa', 'dialogo', '--corpus', str(CORPUS), '--saida', str(SAIDA), '--dispositivo', 'cuda', '--max-segundos', '7200'], cwd=ROOT, check=True)
r = json.loads((SAIDA/'dialogo/relatorio.json').read_text())
print('SFT: passo', r['passo'], '| Melhor:', json.loads((SAIDA/'dialogo/melhor/relatorio.json').read_text())['passo'])
print('Confira no log se terminou, pausou por tempo ou parou pela validação.')

## Avaliar e exportar
Compara geração livre, validação/teste humanos e sintéticos separados, e os 72 turnos reservados do motor integrado. O runtime NumPy é exportado sem precisar instalar Torch no site. Os acertos de recuperação factual são separados dos acertos da geração neural.

Ao terminar, compartilhe `baseline_2m6.json`, `candidato_16m.json` e `contrato_72.json`, além dos relatórios de treino. `numpy/` é o candidato para teste, não uma promoção automática à main.

In [ ]:
subprocess.run([sys.executable, '-u', 'scripts/experimento_transformer_16m.py', '--etapa', 'avaliar', '--corpus', str(CORPUS), '--saida', str(SAIDA)], cwd=ROOT, check=True)
print('Resultados completos:', SAIDA)
for nome in ('baseline_2m6.json', 'candidato_16m.json', 'contrato_72.json'):
    p=SAIDA/nome
    print(nome, p.stat().st_size, 'bytes')